In this code, we will aggregate ESRI-based agroforestry trees from individual trees (1m resolution) to 30m resolution tree canopy cover percentage across Europe and the Sahel to facilitate eventualy the charterisation of agroforestry systems in both regions. In this example, we will use ESRI land use and land cover data particularly the 'croplands' and 'rangelands' classes where agroforestry systems are likely to occur. We filtered the rangelands using a Human Footprint Index to select only rangelands subject to human pressure, serving as a proxy for pastureland. These data will be overlaid with a 1m tree canopy height map to eventually map individual agroforestry trees as Trees on Croplands and Trees on Pastureland.

1- Import and pre-process required data

In [1]:
#Import packages :
import ee
import geemap
# Initilise GEE :
ee.Authenticate()
ee.Initialize(project= 'agroforestery')
#Set the vizualization parameters :
Map = geemap.Map(height='700pt', width='700')

In [2]:
# Define the region of interest (ROI) as Sahel and Or Europe (by chnaging this, you can change the extent where the aggregation will be perfomed)
regions = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
ROI = regions.filter(
ee.Filter.eq("wld_rgn", "Europe")
)
Map.addLayer(ROI, {}, 'Europe')

In [3]:
#Import using GEE assets link : 
canopyHeight = ee.ImageCollection("projects/meta-forest-monitoring-okw37/assets/CanopyHeight").mosaic()
#Convert tree height into tree / no tree binary mask :
Trees_higher_1 = canopyHeight.updateMask(canopyHeight.gte(1))
binary_canopy = Trees_higher_1.mask(Trees_higher_1).gt(0).rename('TreeCoverBinary')
Tolantreecover = binary_canopy.unmask(0)
#Visualize the binary tree cover map on the map :
Map.addLayer(Tolantreecover, {'min': 0, 'max': 1, 'palette': ['red', 'green']}, 'Tolan tree canopy 1m', True)

In [4]:
# Define a functon to select specifi land use of interest from the ESRI LULC dataset and calculate the mode over a specified year range
def get_lulc_class(dataset_collection, year_range, lulc_class_number, class_name):
    
    # Generate year range
    years = list(range(year_range[0], year_range[1]))
    
    # Function to extract class for a given year
    def extract_class(year):
        dataset = (dataset_collection
                   .filterDate(f'{year}-01-01', f'{year}-12-31')
                   .mosaic())
                   
        class_mask = dataset.eq(lulc_class_number).unmask(0)
        return class_mask.set('year', year).set('class', class_name)
    
    # Create collection of class layers
    class_collection = ee.ImageCollection([extract_class(year) for year in years])
    
    # Calculate mode (most frequent pixel value)
    class_mode = class_collection.reduce(ee.Reducer.mode())
    
    # Mask out non-class areas
    class_masked = class_mode.updateMask(class_mode)
    
    return  class_masked

In [ ]:
# Global 100m Terrestrial Human Footprint (HFP-100) : https://doi.org/10.5061/dryad.ttdz08m1f
# A copy of the data was uploaded to a private GEE asset for processing purposes but is not publicly shared
# The data is publicly available from Dryad: https://doi.org/10.5061/dryad.ttdz08m1f
hpf = ee.ImageCollection("Data link").mosaic()
hpf = hpf.updateMask(hpf.gt(4)) # We set the 4 threshold to only keep areas where human pressure is high accoridng to https://doi.org/10.5061/dryad.ttdz08m1f

In [ ]:
# Apply the funtion to croplands ( Id = 5) and rangelands ( Id = 11) for the years 2017 to 2025
esri_lulc = ee.ImageCollection('projects/sat-io/open-datasets/landcover/ESRI_Global-LULC_10m_TS')
ESRI_croplands  = get_lulc_class(esri_lulc, (2017, 2024), 5, 'Cropland')

ESRI_rangelands = get_lulc_class(esri_lulc, (2017, 2024), 11, 'Rangeland')
ESRI_pastures = ESRI_rangelands.updateMask(hpf)

# Visualize the results on the map
Map.addLayer(ESRI_croplands, {'palette': ['#ffff00'], 'min': 0, 'max': 9}, 'ESRI croplands')
Map.addLayer(ESRI_pastures, {'palette': ['#ff8000'], 'min': 0, 'max': 9}, 'ESRI Pasture')

In [7]:
# ESRI based agroforestry map :
# Overaly Trees layer with ESRI croplands 
ESRI_TOC = binary_canopy.updateMask(ESRI_croplands)
# Overaly Trees layer with ESRI pastures
ESRI_TOP = binary_canopy.updateMask(ESRI_pastures)
ESRI_ESRI = ee.ImageCollection([ESRI_TOC, ESRI_TOP]).mosaic()
Map.addLayer(ESRI_ESRI, {'min': 1,'max': 1,'palette': ['#006d2c'] }, 'ESRI Agroforestry Map')

2 - Reproject and aggregate the 1m TOC and TOP: In this case, we will reproject the 1m TOC and TOP data from WGS84 (EPSG:4326) to Sinusoidal Projection (SR-ORG:6974). This is important to calculate the tree cover percentage relative to equal pixel areas across the world.

In [9]:
def reproject_and_aggregate_to_30m(image, output_name):
   
    # Reproject to Sinusoidal projection
    reprojected = image.reproject(
        crs='SR-ORG:6974',
        scale=1,
    )
    
    # Get projection at 30m scale
    projection = ee.Image(reprojected).projection()
    projectionAt30m = projection.atScale(30)
    
    # Aggregate from 1m to 30m using sum reducer
    aggregated = (
        reprojected
        .reduceResolution(
            reducer=ee.Reducer.sum().unweighted(),
            maxPixels=10000
        )
        .reproject(crs=projectionAt30m)
    )
    
    # Convert to percentage (divide by 900 since 30m² = 900 * 1m²)
    result = aggregated.divide(900).multiply(100).rename(output_name)
    
    return result

# Apply the function to both images
TOC_30m = reproject_and_aggregate_to_30m(ESRI_TOC, 'TOC_30m')
Map.addLayer(TOC_30m, {'min': 0, 'max': 100, 'palette': ['#ffffcc', '#41ab5d']}, 'TOC_30m')
TOP_30m = reproject_and_aggregate_to_30m(ESRI_TOP, 'TOP_30m')
Map.addLayer(TOP_30m, {'min': 0, 'max': 100, 'palette': ['#ffffcc', '#41ab5d']}, 'TOP_30m')

3 - Export the data using tiled export: We use this approach to handle memory calculations and export tiles to Google Earth Engine assets one tile at a time. This process can take from many hours to many days depending on the spatial extent. Thsi code were adapated from Spatial Thought (https://spatialthoughts.com/2024/10/23/large-image-exports-gee/)

In [ ]:
# Choose the export CRS
crs = 'SR-ORG:6974'  # Sinusoidal projection
 
# Choose the pixel size for export (meters)
pixelSize = 30
 
# Choose the export tile size (pixels)
tileSize = 10000
 
# Calculate the grid size (meters)
gridSize = tileSize * pixelSize
 
# Create the grid covering the geometry bounds
bounds = ROI.bounds(**{
  'proj': crs, 'maxError': 1
})
 
grid = bounds.coveringGrid(**{
  'proj':crs, 'scale': gridSize
})

# Filter the grid to keep only the cells that intersect with the ROI
filtered_grid = grid.filter(ee.Filter.intersects('.geo', ROI.geometry()))
grid = filtered_grid
Map.addLayer(
    grid , {'color': 'red'}, 'Grid')

In [ ]:
# Calculate the coordinates of the top-left corner of the grid
bounds = grid.geometry().bounds(**{
  'proj': crs, 'maxError': 1
})
 
# Extract the coordinates of the grid
coordList = ee.Array.cat(bounds.coordinates(), 1)
 
xCoords = coordList.slice(1, 0, 1)
yCoords = coordList.slice(1, 1, 2)
 
# We need the coordinates of the top-left pixel
xMin = xCoords.reduce('min', [0]).get([0,0])
yMax = yCoords.reduce('max', [0]).get([0,0])
 
# Create the CRS Transform
 
# The transform consists of 6 parameters:
# [xScale, xShearing, xTranslation, 
#  yShearing, yScale, yTranslation]
transform = ee.List([
    pixelSize, 0, xMin, 0, -pixelSize, yMax]).getInfo()
print(transform)
# Check the number of tiles in the grid
tile_ids = grid.aggregate_array('system:index').getInfo()
print('Total tiles', len(tile_ids))

In [ ]:
# Export each tile to Earth Engine Asset
for i, tile_id in enumerate(tile_ids):
    # Correct way: get the i-th element directly
    feature = ee.Feature(grid.toList(grid.size()).get(i))
    geometry = feature.geometry()

    task_name = 'ROI_Name_' + tile_id.replace(',', '_')

    # Define asset ID (path in your EE assets)
    asset_id = 'Asset_ID_' + task_name
    task = ee.batch.Export.image.toAsset(**{
        'image': TOC_30m,  
        'description': f'TOCTile_{task_name}',
        'assetId': asset_id,
        'region': geometry,
        'maxPixels': 10000000000000,
    })
    
    task.start()
    print('Started Asset Export Task: ', i+1)